# COMP3122 — Lab 09 — Decision Trees and Multi-Class Classification
### Week 11 · Chapter 9

**Applied Data Science**

**This lab is graded.** It covers Chapter 9 in the order the chapter covers it.

**How it works:** each task has a cell for your answer, followed by a cell that checks it. Run the check cell and it prints `PASS`, or shows you what it expected.

**Your numbers are not the book's.** This lab splits every dataset with `test_size=0.25` and `random_state=3122`, and its trees use `random_state=3122`.

**Keep the feature columns in the setup cell's original order.** You may reorder rows if you reorder their labels with them. Changing column order can change how the tree breaks ties, even with the same random_state, so keep the original order for this lab's checks.

**Derive your answers; do not type them.** Several tasks check how you got there as well: the splits are compared row by row, your trees are checked for their settings and scored again, and your confusion matrix is rebuilt from your own tree.

**How it is marked:** your mark is the number of answer checks that print `PASS`, out of **53**. The 3 `GUARD` checks in Task 7.2 earn no marks; they only warn you if the shared data was overwritten. Your professor also reads your code: where a task asks you to calculate or fit something, a result typed in rather than computed does not earn its check. Reading answers (1.3, 3.2), your choices (3.4's last answer, 4.3, 5.2's sentence) and 6.3's class are fine to type.

**Nothing in this lab needs the internet.**

Work in order. The setup cell builds `X_iris`, `X_cancer`, `X_wine` and their targets and defines the checkers, so run it first. **Do not reassign those names** — build your answers into the variables each task asks for. If a check will not pass after five minutes, ask.

---
## Setup — run this first

In [ ]:
# Run this cell once, before you start. It builds the data and defines the checkers.
import os
# explain: silences a harmless CPU-count warning on the course server
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris, load_breast_cancer, load_wine
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import confusion_matrix, precision_score, recall_score

# ---- three datasets from Chapter 9 ----
# No download: all three ship inside scikit-learn.
iris = load_iris(as_frame=True)
# 0 = setosa, 1 = versicolor, 2 = virginica
X_iris, y_iris = iris.data, iris.target
cancer = load_breast_cancer(as_frame=True)
# the ten "mean" columns; 1 = malignant (as in Chapter 8)
X_cancer = cancer.data[[c for c in cancer.data.columns if c.startswith("mean")]]
y_cancer = (cancer.target == 0).astype(int).rename("malignant")
wine = load_wine(as_frame=True)
# three cultivars: 0, 1, 2
X_wine, y_wine = wine.data, wine.target

# Untouched copies, so the Task 7.2 guard can tell whether
# anything was reassigned.
X_IRIS_AT_START = X_iris.copy()
X_CANCER_AT_START = X_cancer.copy()
X_WINE_AT_START = X_wine.copy()
Y_AT_START = (y_iris.copy(), y_cancer.copy(), y_wine.copy())

# Reference splits for the route checks: THIS lab's split,
# test_size=0.25 and random_state=3122, built from the untouched data.
# The checks compare your objects with these, so a number typed in,
# a shuffled label or a skipped step shows up.
_ITR, _ITE, _IYTR, _IYTE = train_test_split(
    X_IRIS_AT_START, Y_AT_START[0], test_size=0.25, random_state=3122)
_CTR, _CTE, _CYTR, _CYTE = train_test_split(
    X_CANCER_AT_START, Y_AT_START[1], test_size=0.25, random_state=3122)
_WTR, _WTE, _WYTR, _WYTE = train_test_split(
    X_WINE_AT_START, Y_AT_START[2], test_size=0.25, random_state=3122)
# The scaler Task 7.1 must match: fitted on the TRAINING wines only.
_SW_REF = StandardScaler().fit(_WTR)


def check(label, got, want):
    """Compare your answer with the expected value. Prints PASS or NOT YET.

    `got` may be a value, or a zero-argument function producing one. The second
    form is used wherever the check has to reach into your answer first — read an
    attribute, round a number, index into an array. Before the answer is filled in
    that would raise, and a traceback where a hint should be is not helpful, so it
    is caught and reported as NOT YET instead.
    """
    if callable(got):
        try:
            got = got()
        except Exception as e:
            print(f"NOT YET  {label}")
            print(f"        your answer raised {type(e).__name__}: {e}")
            return

    def _fail(why=None):
        print(f"NOT YET  {label}")
        if why:
            print(f"        {why}")
        print(f"        you gave: {got!r}")
        print(f"        expected: {want!r}")

    PANDAS = (pd.Series, pd.DataFrame)
    if isinstance(want, PANDAS) or isinstance(got, PANDAS):
        if type(got) is not type(want):
            return _fail(f"wrong type — got {type(got).__name__}, expected {type(want).__name__}")
        if list(got.index) != list(want.index):
            return _fail("the row labels do not match")
        if isinstance(want, pd.DataFrame) and list(got.columns) != list(want.columns):
            return _fail("the column labels do not match")
        try:
            ok = bool(got.equals(want))
        except Exception:
            ok = False
    else:
        if isinstance(got, (bool, np.bool_)) and not isinstance(want, (bool, np.bool_)):
            return _fail("True/False is not a number here")
        g, w = np.asarray(got), np.asarray(want)
        try:
            if g.shape == w.shape and g.dtype.kind in "fc" and w.dtype.kind in "fc":
                ok = bool(np.allclose(g, w))   # decimals: allow a tiny rounding difference
            else:
                ok = bool(np.array_equal(g, w))
        except Exception:
            ok = False

    if ok:
        print(f"PASS  {label}")
    else:
        _fail()


def whole(v):
    """A count or a position must be a whole number. 4 and 4.0 count as 4; 1.9 stays 1.9
    and does not pass. (Turning 1.9 into 1 would let a wrong answer through.)"""
    if isinstance(v, (bool, np.bool_)):
        return v
    if isinstance(v, (int, np.integer)):
        return int(v)
    if isinstance(v, (float, np.floating)) and float(v).is_integer():
        return int(v)
    return v


def _same_rows(got, want):
    """None if `got` holds exactly the rows of `want` (in any order, none twice); else a reason."""
    rows = list(got.index)
    if len(set(rows)) != len(rows):
        return "some rows appear more than once"
    if set(rows) != set(want.index):
        return "not the rows the required split picks"
    return None


def labels_match(got, want):
    """True if `got` is a Series holding the same 0/1 labels as `want`, each on its own row.
    Row ORDER does not matter (sorting is fine); which label sits on which row does.
    Any whole-number or decimal type is fine (int32, int64, Int64, float64 ...).
    True/False values, text, missing values and wrong labels are not. Otherwise returns a short
    reason, which the check prints."""
    if not isinstance(got, pd.Series):
        return f"not a Series (got {type(got).__name__})"
    why = _same_rows(got, want)
    if why:
        return why
    if pd.api.types.is_bool_dtype(got.dtype):
        return "True/False values: make them 1/0 with .astype(int)"
    if not pd.api.types.is_numeric_dtype(got.dtype):
        return f"the values are not numbers (dtype {got.dtype})"
    if got.isna().any():
        return "it has missing values"
    w = want.loc[got.index]
    bad = int((got.to_numpy(dtype=float) != w.to_numpy(dtype=float)).sum())
    return True if bad == 0 else f"{bad} of {len(w)} labels are not on their own rows"


def paired_labels(features, labels, want):
    """True if `labels` is in the SAME row order as `features` (scikit-learn pairs rows by
    position, not by index) and holds the right label on every row. Reordering both together
    is fine; reordering only one of them is not."""
    if not isinstance(features, (pd.DataFrame, pd.Series)) or not isinstance(labels, pd.Series):
        return labels_match(labels, want)
    if not features.index.equals(labels.index):
        return ("the labels are not in the same row order as the features: "
                "put each feature part and its labels in the same order (reorder both together)")
    return labels_match(labels, want)


def rows_match(got, want):
    """True if the DataFrame `got` holds exactly the rows and columns of `want`, with the same
    values. Row order and column order do not matter."""
    if not isinstance(got, pd.DataFrame):
        return f"not a DataFrame (got {type(got).__name__})"
    why = _same_rows(got, want)
    if why:
        return why
    if sorted(got.columns) != sorted(want.columns) or len(set(got.columns)) != len(got.columns):
        return "not the expected columns"
    w = want.loc[got.index, list(got.columns)]
    return bool(np.allclose(got.to_numpy(dtype=float), w.to_numpy(dtype=float)))


def split_ok(X_part, y_part, X_ref, y_ref, fix=""):
    """True if X_part holds the rows of the required split, in the setup cell's ORIGINAL
    column order, and y_part holds each row's own label in the same row order.
    Reordering rows (with their labels) is fine. Changing column order can change
    how the tree breaks ties, even with the same random_state, so it is not accepted
    here. `fix` is the repair line the hint shows, with the task's own names."""
    if isinstance(X_part, pd.DataFrame) and list(X_part.columns) != list(X_ref.columns):
        return ("keep the columns in the setup cell's original order "
                "(changing column order can change the tree's tie-breaking). "
                + (f"Repair:  {fix}" if fix else ""))
    why = rows_match(X_part, X_ref)
    if why is not True:
        return why
    return paired_labels(X_part, y_part, y_ref)


def split_pair_ok(train, test):
    """Both halves of a split: the training result if it fails (its reason kept), else the test result."""
    return train if train is not True else test


def scaled_ok(scaler, A, B, X_tr, X_te):
    """True if `scaler` learned the training wines only and A, B are its transforms
    of the training and test rows (in the student's own row order)."""
    if not hasattr(scaler, "mean_"):
        return "sw is not a fitted StandardScaler"
    if int(scaler.n_samples_seen_) != len(_WTR) or not np.allclose(scaler.mean_, _SW_REF.mean_) \
            or not np.allclose(scaler.var_, _SW_REF.var_):
        return "sw did not learn the TRAINING wines only: fit it on Xw_train"
    want_a = _SW_REF.transform(_WTR.loc[X_tr.index])
    want_b = _SW_REF.transform(_WTE.loc[X_te.index])
    if np.shape(A) != want_a.shape or not np.allclose(np.asarray(A, dtype=float), want_a):
        return "Xw_train_s is not sw's transform of the training wines"
    if np.shape(B) != want_b.shape or not np.allclose(np.asarray(B, dtype=float), want_b):
        return "Xw_test_s is not sw's transform of the test wines"
    return True


def values_match(got, want):
    """True if the array `got` has the shape and values of `want` (tiny rounding allowed)."""
    g = np.asarray(got, dtype=float)
    if g.shape != np.shape(want):
        return f"shape {g.shape}, expected {np.shape(want)}"
    return bool(np.allclose(g, want))


def tree_settings(model, depth, seed=3122):
    """True if `model` is a fitted DecisionTreeClassifier with max_depth=depth and random_state=seed;
    otherwise a short reason, which the check prints."""
    if not isinstance(model, DecisionTreeClassifier):
        return f"not a DecisionTreeClassifier (got {type(model).__name__})"
    if not hasattr(model, "tree_"):
        return "the tree is not fitted yet: call .fit()"
    if model.max_depth != depth or model.random_state != seed:
        return f"needs max_depth={depth}, random_state={seed} (got {model.max_depth}, {model.random_state})"
    return True


def on(model, ref):
    """The reference rows `ref`, in the column order `model` was fitted with."""
    cols = getattr(model, "feature_names_in_", None)
    return ref[list(cols)] if cols is not None else ref


def raises(fn):
    """Run fn() and return the name of the exception it raised, or 'no error'.

    Task 4.3 asks you to find out what a wrong line actually does. Guessing
    is not the same as knowing, and this is how you check.
    """
    try:
        fn()
    except Exception as e:
        return type(e).__name__
    return "no error"


print("Ready. X_iris:", X_iris.shape, " X_cancer:", X_cancer.shape, " X_wine:", X_wine.shape)
import sklearn
print("scikit-learn", sklearn.__version__, " pandas", pd.__version__, " numpy", np.__version__)


---
## Part 1 — A model you can read  ·  §9.1

A rule card, written as a function and tested on this lab's own split.

**Task 1.1.** Split the Iris data (`X_iris`, `y_iris`) into `X_train`, `X_test`, `y_train`,
`y_test` with **`test_size=0.25` and `random_state=3122`**. This is not the book's split.

**Keep the feature columns in the setup cell's original order.** You may reorder rows if you reorder their labels with them. Changing column order can change how the tree breaks ties, even with the same random_state, so keep the original order for this lab's checks.

In [ ]:
X_train, X_test, y_train, y_test = None, None, None, None

In [ ]:
check('1.1 training flowers', lambda: split_ok(X_train, y_train, _ITR, _IYTR, 'X_train = X_train[X_iris.columns]'), True)
check('1.1 test flowers', lambda: split_ok(X_test, y_test, _ITE, _IYTE, 'X_test = X_test[X_iris.columns]'), True)

**Task 1.2.** Here is a rule card:

1. Is the petal length **2.45 cm or less**? If yes, **setosa [0]**.
2. If no: is the petal width **1.75 cm or less**? If yes, **versicolor [1]**; if no, **virginica [2]**.

Write it as a function `card_rule(petal_length, petal_width)` that returns 0, 1 or 2. Then apply it
to every flower in `X_test` and store how many it gets right in `t12_correct`.

In [ ]:
def card_rule(petal_length, petal_width):
    pass

t12_correct = None

In [ ]:
check('1.2 card_rule follows the card (on the lines, and just past them)', lambda: [card_rule(2.45, 0.2), card_rule(2.46, 1.75), card_rule(4.9, 1.76)], [0, 1, 2])
check('1.2 card correct on the 38 test flowers', lambda: whole(t12_correct), 36)

**Task 1.3.** A tree has these rules: *Is the wind above 60 km/h? If yes, "storm". If no, is it above
20 km/h? If yes, "breezy". If no, "calm".*

- `t13_leaves` — how many leaves the tree has
- `t13_depth` — its depth
- `t13_label` — the label for a wind of **exactly 20 km/h** (a string)

In [ ]:
t13_leaves = None
t13_depth = None
t13_label = None

In [ ]:
check('1.3 leaves', lambda: whole(t13_leaves), 3)
check('1.3 depth', lambda: whole(t13_depth), 2)
check('1.3 label at exactly 20 km/h', t13_label, 'calm')

---
## Part 2 — How a tree chooses its questions  ·  §9.2

**Task 2.1.** Write `gini(labels)`: it takes a Series of class labels and returns
**1 − (the sum of the squared class proportions)**.

In [ ]:
def gini(labels):
    pass

In [ ]:
check('2.1 a pure group', lambda: round(float(gini(pd.Series([1, 1, 1, 1]))), 4), 0.0)
check('2.1 a 50/50 group', lambda: round(float(gini(pd.Series([0, 1, 0, 1]))), 4), 0.5)
check('2.1 6 of A, 2 of B', lambda: round(float(gini(pd.Series(['A'] * 6 + ['B'] * 2))), 4), 0.375)

**Task 2.2.** Using `gini` and your training flowers:

- `t22_before` — the Gini impurity of `y_train`, rounded to 4 decimals
- `t22_yes`, `t22_no` — the Gini of the two groups made by *petal length ≤ 2.45*, rounded to 4 decimals

In [ ]:
t22_before = None
t22_yes = None
t22_no = None

In [ ]:
check('2.2 Gini before any question', lambda: round(float(t22_before), 4), 0.6658)
check('2.2 Gini of the yes group', lambda: round(float(t22_yes), 4), 0.0)
check('2.2 Gini of the no group', lambda: round(float(t22_no), 4), 0.5)

**Task 2.3.** Write `split_score(column, threshold)`: the **size-weighted** Gini of the two groups a
question makes on your training flowers, rounded to 4 decimals. Then store:

- `t23_petal` — the score of *petal length ≤ 2.45*
- `t23_sepal` — the score of *sepal width ≤ 3.0*
- `t23_tie` — the score of *petal width ≤ 0.8*

In [ ]:
def split_score(column, threshold):
    pass

t23_petal = None
t23_sepal = None
t23_tie = None

In [ ]:
check('2.3 petal length 2.45', lambda: round(float(t23_petal), 4), 0.3214)
check('2.3 sepal width 3.0', lambda: round(float(t23_sepal), 4), 0.5564)
check('2.3 petal width 0.8 (the tie)', lambda: round(float(t23_tie), 4), 0.3214)

---
## Part 3 — Fitting and reading a tree  ·  §9.3

**Task 3.1.** Fit `tree = DecisionTreeClassifier(max_depth=2, random_state=3122)` on your
training flowers. Store its accuracy on the test flowers in `t31_acc`.

Then compare it with your card from Task 1.2 (36 of 38). Which scored higher? Keep the answer
for Part 3's last question.

In [ ]:
tree = None
t31_acc = None

In [ ]:
check('3.1 tree has the required settings and is fitted', lambda: tree_settings(tree, 2), True)
check('3.1 test accuracy, recomputed from your tree', lambda: (round(tree.score(on(tree, _ITE), _IYTE), 4), round(float(t31_acc), 4)), (0.8684, 0.8684))

**Task 3.2.** Print the tree's rules with `export_text`. Read them and store:

- `t32_root` — the feature the root asks about (its exact column name)
- `t32_threshold` — the root's threshold, as printed
- `t32_second` — the feature the second question asks about

In [ ]:
t32_root = None
t32_threshold = None
t32_second = None

In [ ]:
check('3.2 root feature', t32_root, 'petal length (cm)')
check('3.2 root threshold', lambda: round(float(t32_threshold), 2), 2.6)
check('3.2 second question feature', t32_second, 'petal length (cm)')

**Task 3.3.** A new flower measures `[6.0, 2.9, 4.9, 1.6]` (sepal length, sepal width, petal length,
petal width). Build it as a one-row DataFrame with `X_iris.columns`, and store:

- `t33_label` — the tree's predicted class (a number)
- `t33_proba` — `predict_proba` for it, rounded to 4 decimals, as a list of three numbers

In [ ]:
t33_label = None
t33_proba = None

In [ ]:
check('3.3 predicted class', lambda: whole(t33_label), 2)
check('3.3 probabilities: the class shares of its leaf', lambda: list(np.round(np.asarray(t33_proba, dtype=float).ravel(), 4)), [0.0, 0.0526, 0.9474])

**Task 3.4.** Fit the same tree (`max_depth=2, random_state=3122`) on **scaled** features:
a `StandardScaler` fitted on `X_train` only. Store in `t34_same` how many of the 38 test flowers
get the **same** prediction as `tree`. Then store in `t34_higher` which scored higher on the test
flowers in this lab: `"card"` or `"tree"`.

In [ ]:
t34_same = None
t34_higher = None

In [ ]:
check('3.4 same predictions after scaling', lambda: whole(t34_same), 38)
check('3.4 which scored higher here', t34_higher, 'card')

---
## Part 4 — Trees overfit: the `max_depth` dial  ·  §9.4

Harder data: the breast cancer tumours from Chapter 8 (`X_cancer`, `y_cancer`; 1 = malignant).

**Task 4.1.** Split `X_cancer`, `y_cancer` into `Xc_train`, `Xc_test`, `yc_train`, `yc_test` with
`test_size=0.25` and `random_state=3122`. Then fit one tree per depth in
`[1, 2, 3, 4, 5, 6, 7, 8, None]`, each with `random_state=3122`, and build two dictionaries
keyed by depth: `t41_train` (training accuracy) and `t41_test` (test accuracy), each value
rounded to 4 decimals.

**Keep the feature columns in the setup cell's original order.** You may reorder rows if you reorder their labels with them. Changing column order can change how the tree breaks ties, even with the same random_state, so keep the original order for this lab's checks.

In [ ]:
Xc_train, Xc_test, yc_train, yc_test = None, None, None, None
t41_train = {}
t41_test = {}

In [ ]:
check('4.1 training tumours', lambda: split_ok(Xc_train, yc_train, _CTR, _CYTR, 'Xc_train = Xc_train[X_cancer.columns]'), True)
check('4.1 test tumours', lambda: split_ok(Xc_test, yc_test, _CTE, _CYTE, 'Xc_test = Xc_test[X_cancer.columns]'), True)
check('4.1 training accuracy by depth', lambda: {k: round(float(v), 4) for k, v in t41_train.items()}, {1: 0.9178, 2: 0.9413, 3: 0.9531, 4: 0.9648, 5: 0.9859, 6: 0.9953, 7: 0.9977, 8: 1.0, None: 1.0})
check('4.1 test accuracy by depth', lambda: {k: round(float(v), 4) for k, v in t41_test.items()}, {1: 0.9021, 2: 0.9161, 3: 0.9231, 4: 0.9021, 5: 0.9091, 6: 0.9021, 7: 0.9091, 8: 0.9021, None: 0.9021})

**Task 4.2.** From your dictionaries:

- `t42_perfect` — the **smallest** depth whose training accuracy is 1.0
- `t42_best` — the depth with the **highest test** accuracy
- `t42_gap` — how many test tumours separate that best depth from depth 1 (143 test tumours)

In [ ]:
t42_perfect = None
t42_best = None
t42_gap = None

In [ ]:
check('4.2 smallest depth with training 1.0', lambda: whole(t42_perfect), 8)
check('4.2 depth with the best test accuracy', lambda: whole(t42_best), 3)
check('4.2 tumours between best and depth 1', lambda: whole(t42_gap), 3)

**Task 4.3.** A classmate writes: *"Depth 3 had the best test score, so depth 3 is the right depth for
this data, and 0.9231 is how well it will do on new tumours."* Store the best reply in `t43`, one of:

- `"agree"`
- `"the gap is small, and choosing with the test set spends it"`
- `"deeper is always better"`

In [ ]:
t43 = None

In [ ]:
check('4.3 the classmate\'s claim', t43, 'the gap is small, and choosing with the test set spends it')

---
## Part 5 — Feature importance  ·  §9.5

**Task 5.1.** Fit `t5 = DecisionTreeClassifier(max_depth=3, random_state=3122)` on `Xc_train`,
`yc_train`. From `t5.feature_importances_`, store:

- `t51_top` — the name of the most important feature
- `t51_share` — its importance, rounded to 4 decimals

In [ ]:
t5 = None
t51_top = None
t51_share = None

In [ ]:
check('5.1 t5 has the required settings and is fitted', lambda: tree_settings(t5, 3), True)
check('5.1 most important feature', t51_top, 'mean concave points')
check('5.1 its share', lambda: round(float(t51_share), 4), 0.8261)

**Task 5.2.** Drop that feature: refit the same depth-3 tree on the other nine columns. Store:

- `t52_top` — the new most important feature
- `t52_acc` — the new tree's test accuracy, rounded to 4 decimals
- `t52_claim` — the sentence you could defend, one of:
  `"mean concave points causes malignancy"`, `"this tree relied most on mean concave points"`,
  `"the other nine features are useless"`

In [ ]:
t52_top = None
t52_acc = None
t52_claim = None

In [ ]:
check('5.2 new most important feature', t52_top, 'mean concavity')
check('5.2 new test accuracy', lambda: round(float(t52_acc), 4), 0.9091)
check('5.2 the sentence you can defend', t52_claim, 'this tree relied most on mean concave points')

---
## Part 6 — More than two classes  ·  §9.6

The Wine data (`X_wine`, `y_wine`): three cultivars, 0, 1 and 2.

**Task 6.1.** Split `X_wine`, `y_wine` into `Xw_train`, `Xw_test`, `yw_train`, `yw_test`
(`test_size=0.25`, `random_state=3122`). Fit `wine_tree = DecisionTreeClassifier(max_depth=2,
random_state=3122)` and store its test predictions in `wine_pred` and the confusion matrix in
`t61_cm` (rows are the truth).

**Keep the feature columns in the setup cell's original order.** You may reorder rows if you reorder their labels with them. Changing column order can change how the tree breaks ties, even with the same random_state, so keep the original order for this lab's checks.

In [ ]:
Xw_train, Xw_test, yw_train, yw_test = None, None, None, None
wine_tree = None
wine_pred = None
t61_cm = None

In [ ]:
check('6.1 the wine split', lambda: split_pair_ok(split_ok(Xw_train, yw_train, _WTR, _WYTR, 'Xw_train = Xw_train[X_wine.columns]'), split_ok(Xw_test, yw_test, _WTE, _WYTE, 'Xw_test = Xw_test[X_wine.columns]')), True)
check('6.1 wine_tree has the required settings and is fitted', lambda: tree_settings(wine_tree, 2), True)
check('6.1 wine_pred is the prediction of wine_tree for every test wine, and t61_cm is built from it', lambda: (np.asarray(wine_pred).shape == (len(Xw_test),) and bool((np.asarray(wine_pred) == wine_tree.predict(on(wine_tree, Xw_test))).all()), np.asarray(t61_cm).tolist() == confusion_matrix(yw_test, wine_pred).tolist(), np.asarray(t61_cm).tolist() == [[10, 0, 1], [1, 14, 5], [0, 1, 13]], confusion_matrix(_WYTE, wine_tree.predict(on(wine_tree, _WTE))).tolist() == [[10, 0, 1], [1, 14, 5], [0, 1, 13]]), (True, True, True, True))

**Task 6.2.** From `t61_cm` (not from a scikit-learn function), store, rounded to 4 decimals:

- `t62_recall_1` — the recall of class [1]
- `t62_precision_2` — the precision of class [2]
- `t62_worst` — the most common single mistake, as `[true class, predicted class]`

In [ ]:
t62_recall_1 = None
t62_precision_2 = None
t62_worst = None

In [ ]:
check('6.2 recall of class [1]', lambda: round(float(t62_recall_1), 4), 0.7)
check('6.2 precision of class [2]', lambda: round(float(t62_precision_2), 4), 0.6842)
check('6.2 most common mistake [true, predicted]', lambda: [whole(v) for v in t62_worst], [1, 2])

**Task 6.3.** Store the **macro** and the **weighted** average recall in `t63_macro` and
`t63_weighted` (rounded to 4 decimals), and in `t63_hidden` the class whose low recall the weighted
average gives the most weight to (a number).

In [ ]:
t63_macro = None
t63_weighted = None
t63_hidden = None

In [ ]:
check('6.3 macro-average recall', lambda: round(float(t63_macro), 4), 0.8459)
check('6.3 weighted-average recall', lambda: round(float(t63_weighted), 4), 0.8222)
check('6.3 the class with low recall and the most weight', lambda: whole(t63_hidden), 1)

**Task 6.4.** Use the setup cell's `raises()` helper to find out what `precision_score(yw_test,
wine_pred)` does with three classes, and store its result in `t64_error`. Then store the
macro-average precision in `t64_macro`, rounded to 4 decimals.

In [ ]:
t64_error = None
t64_macro = None

In [ ]:
check('6.4 what the bare call does', t64_error, 'ValueError')
check('6.4 macro-average precision', lambda: round(float(t64_macro), 4), 0.8422)

---
## Part 7 — Stretching a two-class model  ·  §9.7

**Task 7.1.** Scale the wine features: fit a `StandardScaler` called **`sw`** on `Xw_train` only, and
store the scaled training and test wines in **`Xw_train_s`** and **`Xw_test_s`**. Fit
`ovr = OneVsRestClassifier(LogisticRegression())` and `multi = LogisticRegression()` on `Xw_train_s`.
Store:

- `t71_models` — how many two-class models `ovr` built
- `t71_sum` — the sum of `multi`'s three probabilities for the **first** test wine, rounded to 4 decimals
- `t71_agree` — on how many of the 45 test wines the two models predict the same class

In [ ]:
sw = None
Xw_train_s = None
Xw_test_s = None
t71_models = None
t71_sum = None
t71_agree = None

In [ ]:
check('7.1 models built by one-vs-rest', lambda: whole(t71_models), 3)
check('7.1 multinomial probabilities add up to', lambda: round(float(t71_sum), 4), 1.0)
check('7.1 sw learned the training wines only; the models agree on the test wines', lambda: (scaled_ok(sw, Xw_train_s, Xw_test_s, Xw_train, Xw_test), whole(t71_agree)), (True, 45))

**Task 7.2 — GUARD.** Nothing to write.

The check below reads `X_iris`, `X_cancer` and `X_wine` directly and confirms they still hold what
the setup cell put there. It passes on an untouched notebook, so it is not evidence of work — it
exists to fail loudly if something above overwrote the shared data.

In [ ]:
# Nothing to write for this task.

In [ ]:
check('7.2 GUARD X_iris untouched', lambda: X_iris.equals(X_IRIS_AT_START), True)
check('7.2 GUARD X_cancer untouched', lambda: X_cancer.equals(X_CANCER_AT_START), True)
check('7.2 GUARD X_wine untouched', lambda: X_wine.equals(X_WINE_AT_START), True)

---
## Finish

Choose **Kernel → Restart Kernel and Run All Cells** one last time. If every check prints `PASS` from a clean start, your work is reproducible and you are done.

**Two answers to carry into the quiz:**

- A tree is greedy: on this split your two-question card beat it (Tasks 1.2 and 3.4).
- With three classes, say which class or which average: recall 0.85 and 0.82 are both true of one model (Task 6.3).

---
## Your submission line

Run the cell below and paste **the line it prints** into *Lab Exercise 9* on D2L. Nothing else — not the address bar of this notebook, which carries a port number that stops working when your server restarts. The line is built from your account name, so it always works.

In [ ]:
# Run this cell. Copy the line it prints into 'Lab Exercise 9' on D2L. Nothing else.
from pathlib import Path
account = Path.home().name                                   # your gblearn account name
print(f'http://{account}.gblearn.com/python/?open=comp3122/labs/lab09/COMP3122-Lab09-Student.ipynb')